In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "4"

In [2]:
import random

import torch
import numpy as np
from torch.utils.data import Dataset
from torchvision.io import read_image

def scale_and_pad_to_square(image, size=512):
    _, h, w = image.shape
    scale = 512 / max(h, w)
    image = T.functional.resize(image, (int(h*scale), int(w*scale)))
    _, h, w = image.shape
    l = (512 - w) // 2
    r = 512 - w - l
    t = (512 - h) // 2
    b = 512 - h - t
    return T.functional.pad(image, (l, t, r, b))

def transform(sample, train_mode=False):
    if train_mode:
        # random horizontal flip
        if random.random() < 0.5:
            sample["full_masks"] = (
                T.functional.hflip(sample["full_masks"][0]),
                T.functional.hflip(sample["full_masks"][1])
            )
            sample["burn_masks"] = (
                T.functional.hflip(sample["burn_masks"][0]),
                T.functional.hflip(sample["burn_masks"][1])
            )

    # scale and pad to square
    if "full_masks" in sample:
        sample["full_masks"] = (
            scale_and_pad_to_square(sample["full_masks"][0]),
            scale_and_pad_to_square(sample["full_masks"][1])
        )
    if "burn_masks" in sample:
        sample["burn_masks"] = (
            scale_and_pad_to_square(sample["burn_masks"][0]),
            scale_and_pad_to_square(sample["burn_masks"][1])
        )
    return sample

class SAM_Precessed_MHB(Dataset):
    _views_per_burn = 4
    def __init__(self, root_dir, train_mode = False, transform=transform, return_full_masks=True, return_burn_masks=True):
        self.root_dir = root_dir
        self.train_mode = train_mode
        self.transform = transform
        if train_mode:
            self._pairs_per_burn = 1
        else:
            self._pairs_per_burn = self._views_per_burn ** 2
        self.labels = torch.tensor(np.load(os.path.join(root_dir, "labels.npy")), dtype=torch.float32)
        self.return_full_masks = return_full_masks
        self.return_burn_masks = return_burn_masks
        self._len = self.labels.numel() * self._pairs_per_burn
        self._burn_levels = self.labels.shape[1]
        self._pairs_per_human = self._burn_levels * self._pairs_per_burn

    def __len__(self):
        return self._len

    def __getitem__(self, idx):
        human_idx = idx // self._pairs_per_human
        burn_idx = idx % self._pairs_per_human // self._pairs_per_burn
        if self.train_mode:
            front_view_idx = random.randint(0, self._views_per_burn - 1)
            back_view_idx = random.randint(0, self._views_per_burn - 1)
        else:
            front_view_idx = idx % self._pairs_per_human % self._pairs_per_burn // self._views_per_burn
            back_view_idx = idx % self._pairs_per_human % self._pairs_per_burn % self._views_per_burn
        res = {"label": self.labels[human_idx, burn_idx]}
        if self.return_full_masks:
            res["full_masks"] = (
                read_image(os.path.join(self.root_dir, "full_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                read_image(os.path.join(self.root_dir, "full_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        if self.return_burn_masks:
            res["burn_masks"] = (
                read_image(os.path.join(self.root_dir, "burn_masks", "front", f"{human_idx}_{burn_idx}_{front_view_idx}.png")), 
                read_image(os.path.join(self.root_dir, "burn_masks", "back", f"{human_idx}_{burn_idx}_{back_view_idx}.png"))
            )
        if self.transform is not None:
            return self.transform(res, self.train_mode)
        else:
            return res

def data_collator(inputs,*kwargs):
    with torch.no_grad():
        front_full_mask = []
        back_full_mask = []
        front_burn_mask = []
        back_burn_mask = []
        labels = []

        for input in inputs:
            front_full_mask.append(input["full_masks"][0])
            back_full_mask.append(input["full_masks"][1])
            front_burn_mask.append(input["burn_masks"][0])
            back_burn_mask.append(input["burn_masks"][1])
            labels.append(input["label"])

        front_full_mask = torch.stack(front_full_mask, dim=0)
        back_full_mask = torch.stack(back_full_mask, dim=0)
        front_burn_mask = torch.stack(front_burn_mask, dim=0)
        back_burn_mask = torch.stack(back_burn_mask, dim=0)
        labels = torch.stack(labels, dim=0)
        return {
            "front_full_mask": front_full_mask,
            "back_full_mask": back_full_mask,
            "front_burn_mask": front_burn_mask,
            "back_burn_mask": back_burn_mask,
            "labels": labels
        }

In [3]:
from torch.utils.data import Subset

train_set = SAM_Precessed_MHB("sam_precessed_MHB", train_mode=True)
test_set = SAM_Precessed_MHB("sam_precessed_MHB")

train_indices = torch.tensor([list(range(i*625*train_set._pairs_per_human,(i*625+500)*train_set._pairs_per_human)) for i in range(8)]).flatten()
test_indices = torch.tensor([list(range((i*625+500)*test_set._pairs_per_human,(i*625+625)*test_set._pairs_per_human)) for i in range(8)]).flatten()

train_set = Subset(train_set, train_indices)
test_set = Subset(test_set, test_indices)

In [4]:
from typing import Callable, Optional, Union
from torchvision.models.resnet import BasicBlock, Bottleneck, conv1x1

class ResNetBackbone(torch.nn.Module):
    def __init__(
        self,
        block: type[Union[BasicBlock, Bottleneck]],
        layers: list[int],
        zero_init_residual: bool = False,
        groups: int = 1,
        width_per_group: int = 64,
        replace_stride_with_dilation: Optional[list[bool]] = None,
        norm_layer: Optional[Callable[..., torch.nn.Module]] = None,
    ) -> None:
        super().__init__()
        if norm_layer is None:
            norm_layer = torch.nn.BatchNorm2d
        self._norm_layer = norm_layer

        self.inplanes = 64
        self.dilation = 1
        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]
        if len(replace_stride_with_dilation) != 3:
            raise ValueError(
                "replace_stride_with_dilation should be None "
                f"or a 3-element tuple, got {replace_stride_with_dilation}"
            )
        self.groups = groups
        self.base_width = width_per_group
        self.conv1 = torch.nn.Conv2d(1, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = norm_layer(self.inplanes)
        self.relu = torch.nn.ReLU(inplace=True)
        self.maxpool = torch.nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2, dilate=replace_stride_with_dilation[0])
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2, dilate=replace_stride_with_dilation[1])
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2, dilate=replace_stride_with_dilation[2])
        self.avgpool = torch.nn.AdaptiveAvgPool2d((1, 1))

        for m in self.modules():
            if isinstance(m, torch.nn.Conv2d):
                torch.nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (torch.nn.BatchNorm2d, torch.nn.GroupNorm)):
                torch.nn.init.constant_(m.weight, 1)
                torch.nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck) and m.bn3.weight is not None:
                    torch.nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock) and m.bn2.weight is not None:
                    torch.nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(
        self,
        block: type[Union[BasicBlock, Bottleneck]],
        planes: int,
        blocks: int,
        stride: int = 1,
        dilate: bool = False,
    ) -> torch.nn.Sequential:
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = torch.nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(
            block(
                self.inplanes, planes, stride, downsample, self.groups, self.base_width, previous_dilation, norm_layer
            )
        )
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(
                block(
                    self.inplanes,
                    planes,
                    groups=self.groups,
                    base_width=self.base_width,
                    dilation=self.dilation,
                    norm_layer=norm_layer,
                )
            )

        return torch.nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        return x


In [5]:
class BurnAreaNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone_a = ResNetBackbone(BasicBlock, [2, 2, 2, 2])
        self.backbone_b = ResNetBackbone(BasicBlock, [2, 2, 2, 2])
        self.head = torch.nn.Sequential(
            torch.nn.Linear(1024, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, 1),
            torch.nn.Sigmoid()
        )

    def forward(self, front_full_mask, back_full_mask, front_burn_mask, back_burn_mask):
        features = torch.cat([self.backbone_a(front_full_mask), self.backbone_a(back_full_mask)], dim=-1) - torch.cat([self.backbone_b(front_burn_mask), self.backbone_b(back_burn_mask)], dim=-1)
        return self.head(features)

In [6]:
from torchvision.transforms import v2 as T

transform = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.uint8, scale=True),
    T.Resize((224, 224)),
    T.ToDtype(torch.float32,scale=True)
])

class HFCompatibleModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, front_full_mask, back_full_mask, front_burn_mask, back_burn_mask, labels=None, **kwargs):
        pre = self.model(transform(front_full_mask), transform(back_full_mask), transform(front_burn_mask), transform(back_burn_mask))[:, 0]
        if labels is not None:
            loss = (pre - labels).abs().mean()
            return {"loss": loss, "logits": pre}
        else:
            return {"logits": pre}

In [7]:
import time
import transformers

trainer_args = transformers.TrainingArguments(
    num_train_epochs=480,
    output_dir="./logs/" + time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime()),
    dataloader_num_workers=8,
    per_device_train_batch_size=128,
    weight_decay=1e-3,
    logging_strategy="epoch",
    remove_unused_columns=False,
    save_strategy="epoch",
    save_total_limit=1,
    metric_for_best_model="loss",
    greater_is_better=False,
    bf16=True,
    bf16_full_eval=True,
)

compatible_model = HFCompatibleModel(BurnAreaNet()).cuda()

In [8]:
# trainer = transformers.Trainer(
#     model = compatible_model,
#     train_dataset=train_set,
#     data_collator=data_collator,
#     args=trainer_args,
# )

# trainer.train()

In [9]:
from safetensors.torch import load_file

compatible_model.load_state_dict(load_file("./sam_p_MHB_model.safetensors"))

<All keys matched successfully>

In [10]:
from torch.utils.data import DataLoader
from tqdm import tqdm

test_dataloader = DataLoader(test_set, batch_size=128, num_workers=8, collate_fn=data_collator)
compatible_model.eval()
pre_list = []
label_list = []
with torch.no_grad():
    for batch in tqdm(test_dataloader):
        output = compatible_model(batch["front_full_mask"].cuda(), batch["back_full_mask"].cuda(), batch["front_burn_mask"].cuda(), batch["back_burn_mask"].cuda())
        pre_list.append(output["logits"].cpu())
        label_list.append(batch["labels"])

pre_list = torch.cat(pre_list).reshape(1000,8,16)
label_list = torch.cat(label_list).reshape(1000,8,16)

100%|██████████| 1000/1000 [02:27<00:00,  6.76it/s]


In [11]:
from sklearn.metrics import r2_score
print("R2 Score:", r2_score(label_list.flatten(), pre_list.flatten()))

R2 Score: 0.9987480044364929


In [12]:
from sklearn.metrics import classification_report

triclass_label = torch.zeros_like(label_list)
triclass_label[label_list < 0.05] = 0
triclass_label[(label_list >= 0.05) & (label_list < 0.2)] = 1
triclass_label[label_list >= 0.2] = 2
triclass_pre = torch.zeros_like(pre_list)
triclass_pre[pre_list < 0.05] = 0
triclass_pre[(pre_list >= 0.05) & (pre_list < 0.2)] = 1
triclass_pre[pre_list >= 0.2] = 2
print(classification_report(triclass_label.flatten(), triclass_pre.flatten(), digits=3))

              precision    recall  f1-score   support

         0.0      0.968     0.961     0.964     15520
         1.0      0.963     0.969     0.966     32240
         2.0      0.994     0.993     0.993     80240

    accuracy                          0.983    128000
   macro avg      0.975     0.974     0.975    128000
weighted avg      0.983     0.983     0.983    128000



In [13]:
mae = (pre_list-label_list).abs()
print(f"MAE:{mae.mean().item()*100:.2f}, STD:{torch.std(mae).item()*100:.2f}")

MAE:0.68, STD:0.70


In [14]:
mre = (pre_list-label_list).abs()/(label_list)
print(f"MRE:{mre.mean().item()*100:.2f}, STD:{torch.std(mre).item()*100:.2f}")

MRE:4.52, STD:28.52


In [15]:
import pandas as pd
import pingouin as pg

reshaped_pre = pre_list.view(-1, test_set.dataset._pairs_per_burn)
n_subjects, n_raters = reshaped_pre.shape

long_data_list = []
for i in range(n_raters):
    rater_name = f'Observer_{i+1}'
    temp_df = pd.DataFrame({
        'subject': torch.arange(n_subjects),
        'rater': rater_name,
        'rating': reshaped_pre[:, i] * 100
    })
    long_data_list.append(temp_df)

long_data = pd.concat(long_data_list, ignore_index=True)
pg.intraclass_corr(
    data=long_data,
    targets='subject',
    raters='rater',
    ratings='rating'
)

,Type,Description,ICC,F,df1,df2,pval,CI95
0,ICC1,Single raters absolute,0.999593,39305.755253,7999,120000,0.0,"[1.0, 1.0]"
1,ICC2,Single random raters,0.999593,39307.717732,7999,119985,0.0,"[1.0, 1.0]"
2,ICC3,Single fixed raters,0.999593,39307.717732,7999,119985,0.0,"[1.0, 1.0]"
3,ICC1k,Average raters absolute,0.999975,39305.755253,7999,120000,0.0,"[1.0, 1.0]"
4,ICC2k,Average random raters,0.999975,39307.717732,7999,119985,0.0,"[1.0, 1.0]"
5,ICC3k,Average fixed raters,0.999975,39307.717732,7999,119985,0.0,"[1.0, 1.0]"


In [16]:
baseline_pre = torch.load("sam_p_MHB_baseline_pre.pt")
baseline_mae = (baseline_pre - label_list).abs()
baseline_mre = ((baseline_pre - label_list).abs()/label_list)
print(pg.ttest((baseline_mae.view(-1)*100).numpy().tolist(), (mae.view(-1)*100).numpy().tolist(), paired=True))
print(pg.ttest((baseline_mre.view(-1)*100).numpy().tolist(), (mre.view(-1)*100).numpy().tolist(), paired=True))

                 T     dof alternative  p_val          CI95   cohen_d  power  \
T_test  185.639327  127999   two-sided    0.0  [1.78, 1.81]  0.706496    1.0   

       BF10  
T_test  inf  
                T     dof alternative  p_val          CI95   cohen_d  power  \
T_test  95.006924  127999   two-sided    0.0  [7.21, 7.51]  0.326449    1.0   

       BF10  
T_test  inf  


In [17]:
for i in range(8):
    print(f"Burn Level {i+1}:", end=" ")
    print(f"MAE: {mae[:, i].mean().item()*100:.2f}±{torch.std(mae[:, i]).item()*100:.2f}, ", end=" / ")
    print(f"{baseline_mae[:, i].mean().item()*100:.2f}±{torch.std(baseline_mae[:, i]).item()*100:.2f}, ", end="")
    print(f"MRE: {mre[:, i].mean().item()*100:.2f}±{torch.std(mre[:, i]).item()*100:.2f}", end=" / ")
    print(f"{baseline_mre[:, i].mean().item()*100:.2f}±{torch.std(baseline_mre[:, i]).item()*100:.2f}")

Burn Level 1: MAE: 0.23±0.20,  / 0.62±0.56, MRE: 17.39±79.15 / 28.94±24.56
Burn Level 2: MAE: 0.40±0.31,  / 1.27±0.98, MRE: 5.41±4.35 / 16.95±12.80
Burn Level 3: MAE: 0.54±0.43,  / 1.95±1.57, MRE: 3.67±2.98 / 13.13±10.45
Burn Level 4: MAE: 0.66±0.53,  / 2.58±2.21, MRE: 2.64±2.11 / 10.31±8.76
Burn Level 5: MAE: 0.80±0.66,  / 2.95±2.62, MRE: 2.27±1.87 / 8.47±7.55
Burn Level 6: MAE: 0.91±0.79,  / 3.26±3.37, MRE: 2.01±1.72 / 7.23±7.45
Burn Level 7: MAE: 1.09±1.01,  / 3.59±4.31, MRE: 1.77±1.67 / 5.84±7.10
Burn Level 8: MAE: 0.83±0.82,  / 3.59±6.74, MRE: 0.97±0.99 / 4.14±7.59


In [18]:
import json

with open("mass_human_burns/gender.json", "r") as f:
    gender_data = json.load(f)

gender_mask = torch.zeros(5000, dtype=bool)
gender_mask[torch.tensor(gender_data["male"])]=1

test_human_indices = (test_indices // test_set.dataset._pairs_per_human).view([1000,8,16])[:,0,0]
test_gender_mask = gender_mask[test_human_indices]

print(f"Number of males: {test_gender_mask.sum().item()}, Number of females: {(~test_gender_mask).sum().item()}")

print(f"Male:", end=" ")
print(f"MAE: {mae[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(mae[test_gender_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mae[test_gender_mask==1]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(mre[test_gender_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[test_gender_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mre[test_gender_mask==1]).item()*100:.2f}")
print(f"Female:", end=" ")
print(f"MAE: {mae[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(mae[test_gender_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mae[test_gender_mask==0]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(mre[test_gender_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[test_gender_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mre[test_gender_mask==0]).item()*100:.2f}")

print(pg.ttest((mae[test_gender_mask==0].view(-1)*100).numpy().tolist(), (mae[test_gender_mask==1].view(-1)*100).numpy().tolist()))
print(pg.ttest((mre[test_gender_mask==0].view(-1)*100).numpy().tolist(), (mre[test_gender_mask==1].view(-1)*100).numpy().tolist()))

Number of males: 493, Number of females: 507
Male: MAE: 0.68±0.71 / 2.49±3.93, MRE: 4.24±15.97 / 11.87±14.17
Female: MAE: 0.68±0.69 / 2.46±3.08, MRE: 4.78±36.83 / 11.88±14.32
               T            dof alternative     p_val          CI95   cohen_d  \
T_test  1.864593  127673.177265   two-sided  0.062241  [-0.0, 0.01]  0.010428   

           power   BF10  
T_test  0.462303  0.036  
               T           dof alternative     p_val          CI95   cohen_d  \
T_test  3.406559  88999.132889   two-sided  0.000658  [0.23, 0.85]  0.018864   

           power   BF10  
T_test  0.921338  2.087  


In [19]:
age_mask = torch.zeros(1000, dtype=bool)
age_mask[:500] = 1

print(f"Adult:", end=" ")
print(f"MAE: {mae[age_mask==1].mean().item()*100:.2f}±{torch.std(mae[age_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[age_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mae[age_mask==1]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[age_mask==1].mean().item()*100:.2f}±{torch.std(mre[age_mask==1]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[age_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mre[age_mask==1]).item()*100:.2f}")
print(f"Child:", end=" ")
print(f"MAE: {mae[age_mask==0].mean().item()*100:.2f}±{torch.std(mae[age_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mae[age_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mae[age_mask==0]).item()*100:.2f}, ", end="")
print(f"MRE: {mre[age_mask==0].mean().item()*100:.2f}±{torch.std(mre[age_mask==0]).item()*100:.2f}", end=" / ")
print(f"{baseline_mre[age_mask==0].mean().item()*100:.2f}±{torch.std(baseline_mre[age_mask==0]).item()*100:.2f}")

print(pg.ttest((mae[age_mask==0].view(-1)*100).numpy().tolist(), (mae[age_mask==1].view(-1)*100).numpy().tolist()))
print(pg.ttest((mre[age_mask==0].view(-1)*100).numpy().tolist(), (mre[age_mask==1].view(-1)*100).numpy().tolist()))

Adult: MAE: 0.64±0.66 / 2.42±3.10, MRE: 4.10±17.22 / 11.51±13.53
Child: MAE: 0.72±0.73 / 2.53±3.90, MRE: 4.93±36.48 / 12.25±14.93
                T     dof alternative         p_val          CI95   cohen_d  \
T_test  20.905343  127998   two-sided  6.958106e-97  [0.07, 0.09]  0.116864   

        power       BF10  
T_test    1.0  3.372e+92  
               T     dof alternative         p_val          CI95   cohen_d  \
T_test  5.207907  127998   two-sided  1.912774e-07  [0.52, 1.14]  0.029113   

           power      BF10  
T_test  0.999419  4876.813  


In [20]:
for i in range(4):
    pose_mask = torch.zeros(1000, dtype=bool)
    pose_mask[i*125:(i+1)*125] = 1 # Adult
    pose_mask[i*125+500:(i+1)*125+500] = 1 # Child
    print(f"Pose {i+1}:", end=" ")
    print(f"MAE: {mae[pose_mask==1].mean().item()*100:.2f}±{torch.std(mae[pose_mask==1]).item()*100:.2f}", end=" / ")
    print(f"{baseline_mae[pose_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mae[pose_mask==1]).item()*100:.2f}", end=" / ")
    print(f"MRE: {mre[pose_mask==1].mean().item()*100:.2f}±{torch.std(mre[pose_mask==1]).item()*100:.2f}", end=" / ")
    print(f"{baseline_mre[pose_mask==1].mean().item()*100:.2f}±{torch.std(baseline_mre[pose_mask==1]).item()*100:.2f}") 

Pose 1: MAE: 0.76±0.78 / 2.17±1.92 / MRE: 4.40±10.23 / 10.74±12.55
Pose 2: MAE: 0.69±0.77 / 2.44±2.16 / MRE: 4.39±19.76 / 11.55±12.31
Pose 3: MAE: 0.64±0.61 / 2.55±4.18 / MRE: 4.29±14.42 / 12.41±15.34
Pose 4: MAE: 0.64±0.61 / 2.74±4.87 / MRE: 4.98±50.51 / 12.80±16.29


In [21]:
pose_1_mask = torch.zeros(1000, dtype=bool)
pose_1_mask[:125] = 1
pose_1_mask[500:625] = 1
pose_4_mask = torch.zeros(1000, dtype=bool)
pose_4_mask[375:500] = 1
pose_4_mask[875:1000] = 1

print(pg.ttest((mae[pose_1_mask==1].view(-1)*100).numpy().tolist(), (mae[pose_4_mask==1].view(-1)*100).numpy().tolist()))
print(pg.ttest((mre[pose_1_mask==1].view(-1)*100).numpy().tolist(), (mre[pose_4_mask==1].view(-1)*100).numpy().tolist()))

                T    dof alternative          p_val          CI95   cohen_d  \
T_test  22.461439  63998   two-sided  2.670591e-111  [0.11, 0.13]  0.177573   

        power        BF10  
T_test    1.0  1.123e+107  
               T    dof alternative     p_val            CI95   cohen_d  \
T_test -2.009595  63998   two-sided  0.044478  [-1.14, -0.01]  0.015887   

           power   BF10  
T_test  0.519816  0.067  
